# 02.06 章节实践：语音与 OCR 综合实战

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 02.02-02.05 全部小节</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">综合运用 ASR/TTS/OCR/LLM，完成多模态智能应用</td></tr>
</table>

本章实践包含 **4 道** 综合题，前 2 道需本地环境，后 2 道云环境可做。

---

## 综合实践题 1：语音+OCR 智能助手（需本地麦克风+摄像头）

**任务**：持续监听麦克风，说"拍照"时拍照并用 OCR 识别文字、TTS 播报；说"退出"结束。

**提示**：组合 `recognize_audio` + `cv2.VideoCapture` + `ocr_general_text` + `tts_synthesize`。

In [ ]:
# ===== 环境准备：导入依赖 + 定义本章实践所需的全部函数 =====
import os, sys, base64
sys.path.insert(0, os.path.abspath('./src'))
from dotenv import load_dotenv
load_dotenv()

# 语音相关（来自第02章）
from huaweicloud_sis.client.asr_client import AsrCustomizationClient
from huaweicloud_sis.bean.asr_request import AsrCustomShortRequest
from huaweicloud_sis.client.tts_client import TtsCustomizationClient
from huaweicloud_sis.bean.tts_request import TtsCustomRequest
from huaweicloud_sis.utils import io_utils
from huaweicloud_sis.bean.sis_config import SisConfig

# OCR 相关
from huaweicloudsdkcore.auth.credentials import BasicCredentials
from huaweicloudsdkocr.v1.region.ocr_region import OcrRegion
from huaweicloudsdkocr.v1 import OcrClient, RecognizeGeneralTextRequest, GeneralTextRequestBody

# LLM 相关
from openai import OpenAI
from IPython.display import Audio, display

# 凭证
ak = os.getenv('HUAWEI_SIS_AK', '')
sk = os.getenv('HUAWEI_SIS_SK', '')
region = os.getenv('HUAWEI_SIS_REGION', 'cn-east-3')
project_id = os.getenv('HUAWEI_SIS_PROJECT_ID', '')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY', '')


def recognize_audio(audio_path, language='chinese_16k_general'):
    config = SisConfig(); config.set_connect_timeout(10); config.set_read_timeout(10)
    asr_client = AsrCustomizationClient(ak, sk, region, project_id, sis_config=config)
    data = io_utils.encode_file(audio_path)
    req = AsrCustomShortRequest('wav', language, data); req.set_add_punc('yes')
    result = asr_client.get_short_response(req)
    if isinstance(result, dict) and 'result' in result:
        return result['result'].get('text', '')
    return str(result)


def tts_synthesize(text, output_path='./tts_output.wav', voice='chinese_xiaoyu_common'):
    config = SisConfig(); config.set_connect_timeout(10); config.set_read_timeout(10)
    client = TtsCustomizationClient(ak, sk, region, project_id, sis_config=config)
    req = TtsCustomRequest(text)
    req.set_property(voice); req.set_audio_format('wav'); req.set_sample_rate('8000')
    req.set_saved(True); req.set_saved_path(output_path)
    return client.get_ttsc_response(req)


def ocr_general_text(image_path):
    credentials = BasicCredentials(ak, sk)
    client = OcrClient.new_builder().with_credentials(credentials).with_region(OcrRegion.value_of(region)).build()
    with open(image_path, 'rb') as f:
        image_base64 = base64.b64encode(f.read()).decode('utf-8')
    request = RecognizeGeneralTextRequest()
    request.body = GeneralTextRequestBody(image=image_base64)
    return client.recognize_general_text(request)


def chat_with_llm(messages, model='deepseek-v4-flash', temperature=0.7, stream=False):
    llm_client = OpenAI(api_key=deepseek_api_key, base_url='https://api.deepseek.com/v1')
    response = llm_client.chat.completions.create(model=model, messages=messages, temperature=temperature, stream=stream)
    if stream:
        result = ''
        for chunk in response:
            if chunk.choices[0].delta.content:
                result += chunk.choices[0].delta.content
        return result
    return response.choices[0].message.content

print('✅ 环境就绪，4个核心函数已定义: recognize_audio / tts_synthesize / ocr_general_text / chat_with_llm')


In [ ]:
# ===== 综合实践题 1：你的代码（需本地环境）=====
# while True:
#     录音 → ASR → 判断关键词
#     if "拍照" in command: 拍照 → OCR → TTS → 播放
#     elif "退出" in command: break
print("💡 本地：录音→拍照→OCR→TTS\n云环境：用 ./images/clone_example_16k.wav 代替录音，用代码生成图片代替拍照")


---

## 综合实践题 2：语音对话机器人（ASR → LLM → TTS）

**任务**：录音识别用户输入 → LLM 生成回复 → TTS 合成播报，循环对话直到"退出"。

In [ ]:
# ===== 综合实践题 2：你的代码（需本地环境）=====
# messages = [{"role":"system","content":"你是友好助手"}]
# while True:
#     录音 → ASR → 追加user → LLM → 追加assistant → TTS → 播放
print("💡 本地：录音→ASR→LLM→TTS\n云环境：用 ./images/clone_example_16k.wav 代替录音走通流程")


---

## 综合实践题 3：概念题（云环境可做）

解释以下概念及华为云对应 API：
1. ASR 是什么的缩写？作用？华为云对应哪个 API？
2. TTS 是什么的缩写？作用？华为云对应哪个 API？
3. 声音复刻三步骤？
4. 采样率和位深分别影响音频的什么属性？

In [ ]:
# ===== 综合实践题 3：你的回答 =====
print("题1 ASR:", "______")
print("题2 TTS:", "______")
print("题3 复刻三步:", "______")
print("题4 采样率/位深:", "______")


---

## 综合实践题 4：智能语音翻译助手设计（思考题）

设计"用户说中文 → ASR → LLM 翻译英文 → TTS 合成英文语音"的系统，回答：
- (a) 系统流程图（文字描述）
- (b) ASR 识别英文时 property 参数设什么？
- (c) LLM 翻译的 system prompt 怎么设计？
- (d) TTS 合成英文语音要注意什么？

In [ ]:
# ===== 综合实践题 4：你的回答 =====
print("(a) 流程图:______")
print("(b) ASR英文property:______")
print("(c) LLM prompt:______")
print("(d) TTS英文注意:______")


---

## 参考答案

In [ ]:
# 查看答案
!cat ./answer/02.06_chapter_practice/practice1.txt
print("\n" + "="*50 + "\n")
!cat ./answer/02.06_chapter_practice/practice2.txt
print("\n" + "="*50 + "\n")
!cat ./answer/02.06_chapter_practice/practice3.txt
print("\n" + "="*50 + "\n")
!cat ./answer/02.06_chapter_practice/practice4.txt
